<a href="https://colab.research.google.com/github/swaggincoffee-bit/Tesi/blob/main/00_preparazione_dati/01_Preparazione_dati_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 1 — Mount Google Drive + clone repo
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount("/content/drive")

import subprocess, os, sys

REPO_URL  = "https://github.com/swaggincoffee-bit/Tesi.git"
REPO_PATH = "/content/Tesi"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
sys.path.insert(0, REPO_PATH)

BASE      = "/content/drive/MyDrive/Contatori/CSV"
PATH_SAP  = os.path.join(BASE, "Parco Contatori_G4-G6_20251231_SAP.csv")
PATH_BEAM = os.path.join(BASE, "TABELLA_CONTATORE_BEAM.csv")
PATH_LETT = os.path.join(BASE, "TABELLA_LETTURE_BEAM.csv")

OUTPUT_DIR = "/content/drive/MyDrive/Contatori/OUTPUT/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for label, path in [("SAP", PATH_SAP), ("BEAM", PATH_BEAM), ("LETTURE", PATH_LETT)]:
    print(f"  {'✅' if os.path.exists(path) else '❌'} {label}: {path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ✅ SAP: /content/drive/MyDrive/Contatori/CSV/Parco Contatori_G4-G6_20251231_SAP.csv
  ✅ BEAM: /content/drive/MyDrive/Contatori/CSV/TABELLA_CONTATORE_BEAM.csv
  ✅ LETTURE: /content/drive/MyDrive/Contatori/CSV/TABELLA_LETTURE_BEAM.csv


In [32]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 2 — Installazione librerie
# ══════════════════════════════════════════════════════════════════════════════

!pip install -q statsmodels pyarrow


In [33]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 3 — Import
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import numpy as np
from datetime import timedelta

print("✅ Librerie importate correttamente.")

✅ Librerie importate correttamente.


In [34]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 4 — Caricamento dati grezzi
# ══════════════════════════════════════════════════════════════════════════════

def load_csv(path, label):
    for enc in ["utf-8", "latin-1", "cp1252", "utf-8-sig"]:
        try:
            df = pd.read_csv(path, sep=";", encoding=enc, dtype=str, low_memory=False)
            df.columns = df.columns.str.strip()
            print(f"  ✅ {label}: {df.shape[0]:,} righe × {df.shape[1]} colonne  (encoding: {enc})")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Impossibile leggere {path} con nessun encoding provato.")

print("── Caricamento file ──")
df_sap  = load_csv(PATH_SAP,  "SAP")
df_beam = load_csv(PATH_BEAM, "BEAM")
df_lett = load_csv(PATH_LETT, "LETTURE")

df_lett = df_lett.drop(columns=["Unnamed: 6"], errors="ignore")

print("\n── Colonne SAP:", df_sap.columns.tolist())
print("── Colonne BEAM:", df_beam.columns.tolist())
print("── Colonne LETTURE:", df_lett.columns.tolist())


── Caricamento file ──
  ✅ SAP: 712,571 righe × 13 colonne  (encoding: latin-1)
  ✅ BEAM: 761,967 righe × 7 colonne  (encoding: utf-8)
  ✅ LETTURE: 1,048,001 righe × 7 colonne  (encoding: utf-8)

── Colonne SAP: ['04IND - Provincia (cod)', '04IND - Comune (cod)', '04IND - CAP (cod)', '03IMPS - STec - Accessibile (cod)', '03IMPS - PDR/POD - Esterno (cod)', '03IMPS - Stato telegestione (cod)', '03IMPS - Operandi - Cons.Anno PDR (desc)', '11MDEF - Installazione(dta)', '11MDEF - Costruttore (cod)', '11MDEF - Anno Costruzione (cod)', '11MDEF - Materiale (cod)', '11MDEF - Numero Serie (cod)', 'MEAS - PDR (conteggio)']
── Colonne BEAM: ['MATRICOLA CONTATORE', 'PDR', 'MODELLO CONTATORE', 'VERSIONE FIRMWARE', 'Tecnologia di comunicazione', 'Data ultima comunicazione', 'Data ultima misura']
── Colonne LETTURE: ['PDR', 'Matricola Contatore', 'Timestamp comunicazione', 'Data lettura', 'Stato lettura', 'Diagnostica Contatore']


In [35]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 5 — Ispezione
# ══════════════════════════════════════════════════════════════════════════════

for label, df in [("SAP", df_sap), ("BEAM", df_beam), ("LETTURE", df_lett)]:
    print("═" * 60)
    print(f"{label} — prime 3 righe")
    display(df.head(3))


════════════════════════════════════════════════════════════
SAP — prime 3 righe


,04IND - Provincia (cod),04IND - Comune (cod),04IND - CAP (cod),03IMPS - STec - Accessibile (cod),03IMPS - PDR/POD - Esterno (cod),03IMPS - Stato telegestione (cod),03IMPS - Operandi - Cons.Anno PDR (desc),11MDEF - Installazione(dta),11MDEF - Costruttore (cod),11MDEF - Anno Costruzione (cod),11MDEF - Materiale (cod),11MDEF - Numero Serie (cod),MEAS - PDR (conteggio)
0,Genova,Avegno,16036,ACCESSIBILE,03270006541254,GT,705,30.09.2022,METERSIT,2022,000000000001072798,MTSB030F06147670,1
1,Genova,Avegno,16036,ACCESSIBILE,03270006545605,GT,1110,07.10.2022,METERSIT,2022,000000000001072798,MTSB030F06147565,1
2,Genova,Avegno,16036,ACCESSIBILE,03270006547423,GT,615,25.11.2022,METERSIT,2022,000000000001072798,MTSB030F06147637,1


════════════════════════════════════════════════════════════
BEAM — prime 3 righe


,MATRICOLA CONTATORE,PDR,MODELLO CONTATORE,VERSIONE FIRMWARE,Tecnologia di comunicazione,Data ultima comunicazione,Data ultima misura
0,SMGR034016059502,15441000054940,RSE/2001 LA G4,229,RF,04-FEB-26 05.35.49.000000000 PM,03-FEB-26 12.00.00.000000000 AM
1,SMGR034016679960,15441000054953,RSE/2001 LA G4,230,RF,04-FEB-26 07.26.34.000000000 AM,03-FEB-26 12.00.00.000000000 AM
2,SMGR034017243004,15441000054559,RSE/2001 LA G4,212,RF,04-FEB-26 07.38.00.000000000 AM,04-FEB-26 12.00.00.000000000 AM


════════════════════════════════════════════════════════════
LETTURE — prime 3 righe


,PDR,Matricola Contatore,Timestamp comunicazione,Data lettura,Stato lettura,Diagnostica Contatore
0,15441000040111,MTSB036406264806,17-DEC-25 10.52.56.156000000 AM,16-DEC-25 12.00.00.000000000 AM,ELA,0000000000000000
1,15441000056279,MTSB036430056190,17-DEC-25 09.43.51.718000000 AM,16-DEC-25 12.00.00.000000000 AM,ELA,0000000000000000
2,03270015229272,MTSB036400928751,17-DEC-25 12.54.15.833000000 PM,16-DEC-25 12.00.00.000000000 AM,ELA,0000000000000000


In [36]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 6 — Import mapping colonne da config
# ══════════════════════════════════════════════════════════════════════════════

from src.config import (
    COL_PROV, COL_COMUNE, COL_CAP, COL_ACCESSIBILE,
    COL_TELEGESTIONE, COL_CONSUMO, COL_DATA_INST,
    COL_COSTRUTTORE, COL_ANNO_COSTR, COL_MATERIALE,
    KEY_SAP, KEY_BEAM, KEY_LETT, COL_MODELLO,
    COL_FIRMWARE, COL_TECN_COM, COL_ULT_COM,
    COL_ULT_MIS, COL_DATA_LETT, COL_STATO_LETT,
    COL_DIAGNOSTICA,
)

# Verifica colonne chiave
for df, col, label in [
    (df_sap,  KEY_SAP,       "SAP — matricola"),
    (df_beam, KEY_BEAM,      "BEAM — matricola"),
    (df_lett, KEY_LETT,      "LETTURE — matricola"),
    (df_lett, COL_DATA_LETT, "LETTURE — data lettura"),
    (df_sap,  COL_PROV,      "SAP — provincia"),
]:
    ok = col in df.columns
    print(f"  {'✅' if ok else '❌  DA CORREGGERE'} {label}: '{col}'")

  ✅ SAP — matricola: '11MDEF - Numero Serie (cod)'
  ✅ BEAM — matricola: 'MATRICOLA CONTATORE'
  ✅ LETTURE — matricola: 'Matricola Contatore'
  ✅ LETTURE — data lettura: 'Data lettura'
  ✅ SAP — provincia: '04IND - Provincia (cod)'


In [37]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 7 — Filtraggio Reggio Emilia
# ══════════════════════════════════════════════════════════════════════════════

print("\nValori unici colonna provincia (SAP):")
print(df_sap[COL_PROV].value_counts().to_string())

PROV_TARGET = "Reggio nell'Emilia"
df_sap_re = df_sap[df_sap[COL_PROV].str.contains(PROV_TARGET, case=False, na=False)].copy()

print(f"\nContatori SAP — provincia RE: {len(df_sap_re):,}")
if len(df_sap_re) == 0:
    print("⚠️  Nessun contatore trovato. Controlla PROV_TARGET.")



Valori unici colonna provincia (SAP):
04IND - Provincia (cod)
Genova                299192
Reggio nell'Emilia    228465
Parma                 159463
Savona                 15934
Piacenza                9517

Contatori SAP — provincia RE: 228,465


In [38]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 8 — Merge anagrafica e parco contatori RE
# ══════════════════════════════════════════════════════════════════════════════

df_sap_re["_key"] = df_sap_re[KEY_SAP].str.strip().str.upper()
df_beam["_key"]   = df_beam[KEY_BEAM].str.strip().str.upper()
df_lett["_key"]   = df_lett[KEY_LETT].str.strip().str.upper()

print(f"Contatori unici SAP-RE  : {df_sap_re['_key'].nunique():,}")
print(f"Contatori unici BEAM    : {df_beam['_key'].nunique():,}")
print(f"Contatori unici LETTURE : {df_lett['_key'].nunique():,}")

df_anagrafica = df_sap_re.merge(df_beam, on="_key", how="inner", suffixes=("_sap", "_beam"))

n_prima = len(df_anagrafica)
df_anagrafica = df_anagrafica.drop_duplicates(subset="_key", keep="first")
n_dedup = n_prima - len(df_anagrafica)
if n_dedup > 0:
    print(f"\n⚠️  Rimossi {n_dedup} duplicati di _key dopo il merge SAP↔BEAM")

print(f"\nDopo merge SAP↔BEAM (inner) : {len(df_anagrafica):,} contatori")
print(f"Scartati per anagrafica incompleta: {df_sap_re['_key'].nunique() - len(df_anagrafica):,}")

df_lett_re = df_lett[df_lett["_key"].isin(df_anagrafica["_key"])].copy()

matricole_parco      = set(df_anagrafica["_key"])
matricole_con_lett   = set(df_lett_re["_key"])
matricole_senza_lett = matricole_parco - matricole_con_lett
n_tot, n_con, n_senza = len(matricole_parco), len(matricole_con_lett), len(matricole_senza_lett)

print(f"\nParco contatori RE totale     : {n_tot:,}")
print(f"  ├─ Con almeno una lettura   : {n_con:,}  ({n_con/n_tot*100:.1f}%)")
print(f"  └─ Senza alcuna lettura     : {n_senza:,}  ({n_senza/n_tot*100:.1f}%)")
print(f"\nLetture filtrate (RE)         : {len(df_lett_re):,} righe")


Contatori unici SAP-RE  : 228,465
Contatori unici BEAM    : 761,940
Contatori unici LETTURE : 523,177

⚠️  Rimossi 5 duplicati di _key dopo il merge SAP↔BEAM

Dopo merge SAP↔BEAM (inner) : 226,702 contatori
Scartati per anagrafica incompleta: 1,763

Parco contatori RE totale     : 226,702
  ├─ Con almeno una lettura   : 157,746  (69.6%)
  └─ Senza alcuna lettura     : 68,956  (30.4%)

Letture filtrate (RE)         : 298,745 righe


In [39]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 9 — Parsing date Oracle (vettorizzato)
# ══════════════════════════════════════════════════════════════════════════════

df_lett_re["data_lettura"] = pd.to_datetime(
    df_lett_re[COL_DATA_LETT].str.strip().str.split(" ").str[0],
    format="%d-%b-%y", errors="coerce"
)

n_nat = df_lett_re["data_lettura"].isna().sum()
print(f"Parsate correttamente : {df_lett_re['data_lettura'].notna().sum():,}")
print(f"Non parsabili         : {n_nat:,}")

df_lett_re = df_lett_re.dropna(subset=["data_lettura"])

DATA_MIN = df_lett_re["data_lettura"].min()
DATA_MAX = df_lett_re["data_lettura"].max()
DURATA   = (DATA_MAX - DATA_MIN).days

print(f"Prima lettura : {DATA_MIN.date()}")
print(f"Ultima lettura: {DATA_MAX.date()}")
print(f"Durata dataset: {DURATA} giorni ({DURATA/30:.1f} mesi)")


Parsate correttamente : 298,745
Non parsabili         : 0
Prima lettura : 2025-12-06
Ultima lettura: 2025-12-22
Durata dataset: 16 giorni (0.5 mesi)


In [40]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 10 — Target a finestre di 3 giorni (vettorizzato)
# ══════════════════════════════════════════════════════════════════════════════

WINDOW_DAYS = 3

finestre = []
t = DATA_MIN
while t <= DATA_MAX:
    finestre.append((t, t + timedelta(days=WINDOW_DAYS - 1)))
    t += timedelta(days=WINDOW_DAYS)

print(f"Finestre costruite: {len(finestre)}")
for i, (ts, te) in enumerate(finestre):
    print(f"  W{i+1}: {ts.date()} → {te.date()}")

bins = [ts for ts, _ in finestre] + [finestre[-1][1] + timedelta(days=1)]
df_lett_re["finestra"] = pd.cut(
    df_lett_re["data_lettura"], bins=bins,
    labels=range(1, len(finestre) + 1), right=False
).astype("Int64")

comunicati = (
    df_lett_re.dropna(subset=["finestra"])
    .drop_duplicates(subset=["_key", "finestra"])[["_key", "finestra"]]
    .assign(silente=0)
)

meta = pd.DataFrame({
    "finestra": range(1, len(finestre) + 1),
    "t_start":  [ts for ts, _ in finestre],
    "t_end":    [te for _, te in finestre],
})

idx = pd.MultiIndex.from_product(
    [df_anagrafica["_key"].values, range(1, len(finestre) + 1)],
    names=["_key", "finestra"]
)
df_target = pd.DataFrame(index=idx).reset_index()
df_target = df_target.merge(comunicati, on=["_key", "finestra"], how="left")
df_target["silente"] = df_target["silente"].fillna(1).astype(int)
df_target = df_target.merge(meta, on="finestra", how="left")

print(f"\nShape dataset target: {df_target.shape}")
ct = df_target["silente"].value_counts()
print(f"  Comunicato (0): {ct.get(0,0):,}  ({ct.get(0,0)/len(df_target)*100:.1f}%)")
print(f"  Silente    (1): {ct.get(1,0):,}  ({ct.get(1,0)/len(df_target)*100:.1f}%)")


Finestre costruite: 6
  W1: 2025-12-06 → 2025-12-08
  W2: 2025-12-09 → 2025-12-11
  W3: 2025-12-12 → 2025-12-14
  W4: 2025-12-15 → 2025-12-17
  W5: 2025-12-18 → 2025-12-20
  W6: 2025-12-21 → 2025-12-23

Shape dataset target: (1360212, 5)
  Comunicato (0): 192,400  (14.1%)
  Silente    (1): 1,167,812  (85.9%)


In [41]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 11 — Feature engineering + cross-section + salvataggio parquet
# ══════════════════════════════════════════════════════════════════════════════

from src import utils

# ── Cross-section: collassa panel → una riga per contatore ───────────────────
df_cs = utils.feat_aggregate(df_target, df_anagrafica, df_lett_re)

# ── Feature engineering (variabili numeriche + alias leggibili) ───────────────
df_cs = utils.feat_engineer(df_cs, DATA_MAX)

print(f"df_cs shape: {df_cs.shape}")
print(f"\nVariabili numeriche costruite:")
for col in ["anni_da_costruzione", "consumo_annuo", "gg_da_installazione",
            "firmware_num", "gg_da_ult_com", "gg_da_ult_mis"]:
    n_ok = df_cs[col].notna().sum()
    print(f"  {col:30s}: {n_ok:,} valori  ({df_cs[col].mean():.1f} media)")

print(f"\nDistribuzione target:")
print(f"  pct_silente media        : {df_cs['pct_silente'].mean():.3f}")
print(f"  silente_prevalente = 1   : {df_cs['silente_prevalente'].sum():,}  ({df_cs['silente_prevalente'].mean()*100:.1f}%)")

# ── Salvataggio parquet su Drive ──────────────────────────────────────────────
PROC = OUTPUT_DIR
df_anagrafica.to_parquet(PROC + "df_anagrafica.parquet", index=False)
df_lett_re   .to_parquet(PROC + "df_lett_re.parquet",    index=False)
df_target    .to_parquet(PROC + "df_target.parquet",     index=False)
df_cs        .to_parquet(PROC + "df_cs.parquet",         index=False)

print("\n✅ Parquet salvati in:", PROC)
for f in ["df_anagrafica.parquet", "df_lett_re.parquet", "df_target.parquet", "df_cs.parquet"]:
    size = os.path.getsize(PROC + f) / 1024**2
    print(f"  {f}: {size:.1f} MB")

TypeError: feat_aggregate() takes 2 positional arguments but 3 were given

In [ ]:
display(df_target.sample(20))

,_key,finestra,silente,t_start,t_end
182981,MTSB036506931793,6,1,2025-12-21,2025-12-23
460250,MTSB036402145577,3,1,2025-12-12,2025-12-14
581768,MTSB036405343170,3,1,2025-12-12,2025-12-14
219556,MTSB036501782163,5,1,2025-12-18,2025-12-20
1037560,FIOR034021092508,5,1,2025-12-18,2025-12-20
23504,MTSB036501488885,3,0,2025-12-12,2025-12-14
635998,MTSB036504148928,5,1,2025-12-18,2025-12-20
656859,MTSB033600695290,4,1,2025-12-15,2025-12-17
232896,MTSB036403019313,1,1,2025-12-06,2025-12-08
1015431,SMGR034017252807,4,0,2025-12-15,2025-12-17


In [ ]:
df_lett_re[COL_STATO_LETT].value_counts()

,count
Stato lettura,
ELA,155707
ERR,143038
